# FLX-Nano — Colab Training Runner

Thin execution wrapper for GPU training. Code lives in GitHub, compute happens here, state persists in Drive.

**Runtime → Change runtime type → GPU (A100 or L4 preferred, T4 for smoke tests only)**

In [ ]:
# Cell 1: Clone repo and install
!git clone https://github.com/Unseengap/flux-model.git
%cd flux-model
!pip install -e '.[dev]' -q
!pip install datasets transformers -q

In [ ]:
# Cell 2: Mount Google Drive for persistent .flx state
from google.colab import drive
drive.mount('/content/drive')

FLX_HUB = '/content/drive/MyDrive/flx_state'
import os
os.makedirs(FLX_HUB, exist_ok=True)
print(f'State hub: {FLX_HUB}')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Tokenizer + data processing utilities
import itertools, pickle, os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
VOCAB_SIZE = tokenizer.vocab_size  # 32000
SEQ_LEN = 512
print(f"Tokenizer ready — vocab_size={VOCAB_SIZE}")

DATA_DIR = f"{FLX_HUB}/processed_data"
os.makedirs(DATA_DIR, exist_ok=True)

class CausalLMDataset(Dataset):
    """Raw text → (input_ids, targets) for causal LM training."""
    def __init__(self, examples):
        self.examples = examples

    @classmethod
    def from_texts(cls, texts, tokenizer, max_len=512):
        examples = []
        for text in texts:
            ids = tokenizer(text, truncation=True, max_length=max_len + 1,
                            return_tensors="pt")["input_ids"][0]
            if len(ids) < 2:
                continue
            input_ids = ids[:-1]
            targets = ids[1:]
            pad_len = max_len - len(input_ids)
            if pad_len > 0:
                input_ids = F.pad(input_ids, (0, pad_len), value=tokenizer.pad_token_id)
                targets = F.pad(targets, (0, pad_len), value=-100)
            examples.append((input_ids, targets))
        return cls(examples)

    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

def save_processed(data, path):
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"Saved → {path} ({os.path.getsize(path) / 1e6:.1f} MB)")

def load_processed(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(f"Loaded cached ← {path}")
    return data

print("Data utilities ready")

In [ ]:
# Cell 4: Create FLX-Nano model
from flx.model import FLXNano
from flx.router import ThalamicRouter
from flx.thermal import ThermalEstimator
from flx.bridges import build_bridges
from flx.memory import MemoryController, EpisodicCompressor
from flx.meta_gen import MetaDeltaGenerator
from flx.serialization import save_flx, load_flx

model = FLXNano()
router = ThalamicRouter(d_model=512, cortex_names=model.cortex_names)
model.attach_router(router)

counts = model.count_parameters()
print('Parameter counts:')
for k, v in counts.items():
    print(f'  {k}: {v:,}')

## Phase 0 & 1 — Domain-Diverse Pretraining Data

Phases 0 and 1 share the same dataset: a domain-diverse mix of language, math, code, science, and reasoning text.
Streams from HuggingFace, tokenizes, and saves to Drive. Skips processing if already cached.

In [ ]:
# Process Phase 0/1 data — domain-diverse pretraining mix
# Adjust NUM_SAMPLES_PER_DOMAIN to control dataset size
NUM_SAMPLES_PER_DOMAIN = 20_000  # 100k total across 5 domains
BATCH_SIZE = 16
CACHE_PATH = f"{DATA_DIR}/phase01_pretrain.pkl"

if os.path.exists(CACHE_PATH):
    pretrain_dataset = load_processed(CACHE_PATH)
    pretrain_dataset = CausalLMDataset(pretrain_dataset)
else:
    print("Processing Phase 0/1 data from HuggingFace (this takes a while first time)...")
    all_texts = []

    # Language — general web text
    print("  Downloading language data...")
    ds_lang = load_dataset("cerebras/SlimPajama-627B", split="train", streaming=True)
    lang_texts = [x["text"] for x in itertools.islice(ds_lang, NUM_SAMPLES_PER_DOMAIN)]
    all_texts.extend(lang_texts)
    print(f"    → {len(lang_texts)} language samples")

    # Math
    print("  Downloading math data...")
    ds_math = load_dataset("open-web-math/open-web-math", split="train", streaming=True)
    math_texts = [x["text"] for x in itertools.islice(ds_math, NUM_SAMPLES_PER_DOMAIN)]
    all_texts.extend(math_texts)
    print(f"    → {len(math_texts)} math samples")

    # Code
    print("  Downloading code data...")
    ds_code = load_dataset("bigcode/starcoderdata", data_dir="python", split="train", streaming=True)
    code_texts = [x["content"] for x in itertools.islice(ds_code, NUM_SAMPLES_PER_DOMAIN)]
    all_texts.extend(code_texts)
    print(f"    → {len(code_texts)} code samples")

    # Science
    print("  Downloading science data...")
    ds_sci = load_dataset("allenai/peS2o", "v2", split="train", streaming=True)
    sci_texts = [x["text"] for x in itertools.islice(ds_sci, NUM_SAMPLES_PER_DOMAIN)]
    all_texts.extend(sci_texts)
    print(f"    → {len(sci_texts)} science samples")

    # Reasoning
    print("  Downloading reasoning data...")
    ds_reason = load_dataset("kaist-ai/CoT-Collection", split="train", streaming=True)
    reason_texts = [
        f"{x['source']}\n{x['rationale']}\n{x['target']}"
        for x in itertools.islice(ds_reason, NUM_SAMPLES_PER_DOMAIN)
    ]
    all_texts.extend(reason_texts)
    print(f"    → {len(reason_texts)} reasoning samples")

    import random
    random.shuffle(all_texts)
    print(f"Total: {len(all_texts)} samples. Tokenizing...")

    pretrain_dataset = CausalLMDataset.from_texts(all_texts, tokenizer, max_len=SEQ_LEN)
    save_processed(pretrain_dataset.examples, CACHE_PATH)

pretrain_loader = DataLoader(pretrain_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Phase 0/1 DataLoader ready: {len(pretrain_dataset)} examples, {len(pretrain_loader)} batches")

In [ ]:
# Phase 0 training — Cortex Specialization
from flx.training.phase0_cortex import train_phase0

history = train_phase0(
    model, pretrain_loader,
    num_epochs=10, lr=1e-4,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase0.flx')
print(f"Phase 0 done — {len(history)} steps, final loss: {history[-1]['total_loss']:.4f}")

## Phase 1 — Delta-Receptive Pretraining

Uses the same domain-diverse data as Phase 0. Router is frozen; cortex bases learn to accept delta modifications.

In [ ]:
# Phase 1 training — Delta-Receptive Pretraining
from flx.training.phase1_delta import train_phase1

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase0.flx', device=DEVICE)
history = train_phase1(
    model, pretrain_loader,
    num_epochs=20, lr=5e-5,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase1.flx')
print(f"Phase 1 done — {len(history)} steps, final loss: {history[-1]['total_loss']:.4f}")

## Phase 2 — Thermal Routing + Bridges

Needs **difficulty-stratified** data so τ learns when to think harder vs. coast.
Mix of easy (simple text), medium (grade-school math, basic code), and hard (competition math/code).

In [ ]:
# Process Phase 2 data — difficulty-stratified mix
CACHE_PATH_P2 = f"{DATA_DIR}/phase2_thermal.pkl"

if os.path.exists(CACHE_PATH_P2):
    thermal_dataset = load_processed(CACHE_PATH_P2)
    thermal_dataset = CausalLMDataset(thermal_dataset)
else:
    print("Processing Phase 2 data (difficulty-stratified)...")
    all_texts = []

    # EASY — simple wiki / news text (low τ targets)
    print("  Easy: simple web text...")
    ds_easy = load_dataset("cerebras/SlimPajama-627B", split="train", streaming=True)
    easy_texts = [x["text"] for x in itertools.islice(ds_easy, 15_000)]
    all_texts.extend(easy_texts)
    print(f"    → {len(easy_texts)} easy samples")

    # MEDIUM — grade school math + basic Python
    print("  Medium: GSM8K + MBPP...")
    ds_gsm = load_dataset("openai/gsm8k", "main", split="train")
    gsm_texts = [f"Problem: {x['question']}\nSolution: {x['answer']}" for x in ds_gsm]
    all_texts.extend(gsm_texts)
    print(f"    → {len(gsm_texts)} GSM8K samples")

    ds_mbpp = load_dataset("google-research-datasets/mbpp", "sanitized", split="train")
    mbpp_texts = [f"# {x['prompt']}\n{x['code']}" for x in ds_mbpp]
    all_texts.extend(mbpp_texts)
    print(f"    → {len(mbpp_texts)} MBPP samples")

    # HARD — competition math + hard code (high τ targets)
    print("  Hard: MATH + code contests...")
    ds_math = load_dataset("lighteval/MATH", "all", split="train")
    math_texts = [f"Problem: {x['problem']}\nSolution: {x['solution']}" for x in ds_math]
    all_texts.extend(math_texts)
    print(f"    → {len(math_texts)} MATH samples")

    ds_hard_code = load_dataset("deepmind/code_contests", split="train")
    hard_code_texts = [
        f"# {x['description'][:500]}\n{x['solutions']['solution'][0] if x['solutions']['solution'] else ''}"
        for x in ds_hard_code if x['solutions']['solution']
    ][:5_000]
    all_texts.extend(hard_code_texts)
    print(f"    → {len(hard_code_texts)} code contest samples")

    import random
    random.shuffle(all_texts)
    print(f"Total: {len(all_texts)} samples. Tokenizing...")

    thermal_dataset = CausalLMDataset.from_texts(all_texts, tokenizer, max_len=SEQ_LEN)
    save_processed(thermal_dataset.examples, CACHE_PATH_P2)

thermal_loader = DataLoader(thermal_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Phase 2 DataLoader ready: {len(thermal_dataset)} examples, {len(thermal_loader)} batches")

In [ ]:
# Phase 2 training — Thermal Routing + Bridges
from flx.training.phase2_thermal import train_phase2

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase1.flx', device=DEVICE)
thermal = ThermalEstimator(d_model=512)
model.attach_thermal(thermal)
bridges = build_bridges(model.cortex_names, d_model=512)
model.attach_bridges(bridges)

history = train_phase2(
    model, thermal_loader,
    num_epochs=5, lr=3e-5,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase2.flx')
print(f"Phase 2 done — {len(history)} steps, final loss: {history[-1]['total_loss']:.4f}")

## Phase 3 — Memory System

Needs **multi-turn conversation chains** so the memory controller learns to compress and retrieve across turns.
Converts HF conversation format → `list[list[tuple[Tensor, Tensor]]]`.

In [ ]:
# Process Phase 3 data — multi-turn conversations
NUM_CONVERSATIONS = 5_000
CACHE_PATH_P3 = f"{DATA_DIR}/phase3_conversations.pkl"

if os.path.exists(CACHE_PATH_P3):
    conversation_data = load_processed(CACHE_PATH_P3)
else:
    print("Processing Phase 3 conversation data...")

    # UltraChat — large multi-turn dialogue dataset
    ds_conv = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)

    conversation_data = []
    skipped = 0
    for conv in itertools.islice(ds_conv, NUM_CONVERSATIONS * 2):  # oversample, some get skipped
        messages = conv.get("messages", [])
        if len(messages) < 2:
            skipped += 1
            continue

        turns = []
        for msg in messages:
            text = msg["content"]
            ids = tokenizer(text, truncation=True, max_length=SEQ_LEN + 1,
                            return_tensors="pt")["input_ids"]
            if ids.shape[1] < 2:
                continue
            input_ids = ids[:, :-1]  # [1, seq]
            targets = ids[:, 1:]     # [1, seq]
            # Pad to uniform length
            pad_len = SEQ_LEN - input_ids.shape[1]
            if pad_len > 0:
                input_ids = F.pad(input_ids, (0, pad_len), value=tokenizer.pad_token_id)
                targets = F.pad(targets, (0, pad_len), value=-100)
            turns.append((input_ids, targets))

        if len(turns) >= 2:
            conversation_data.append(turns)

        if len(conversation_data) >= NUM_CONVERSATIONS:
            break

    print(f"Processed {len(conversation_data)} conversations ({skipped} skipped)")
    save_processed(conversation_data, CACHE_PATH_P3)

print(f"Phase 3 data ready: {len(conversation_data)} conversations, "
      f"avg {sum(len(c) for c in conversation_data) / len(conversation_data):.1f} turns each")

In [ ]:
# Phase 3 training — Memory System
from flx.training.phase3_memory import train_phase3

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase2.flx', device=DEVICE)
mem_ctrl = MemoryController(d_model=512)
model.attach_memory(mem_ctrl)
compressor = EpisodicCompressor(d_model=512)

history = train_phase3(
    model, compressor, conversation_data,
    num_epochs=5, lr=2e-5,
    device=DEVICE, log_every=10,
)
save_flx(model, f'{FLX_HUB}/nano_phase3.flx')
print(f"Phase 3 done — {len(history)} steps, final loss: {history[-1]['pred_loss']:.4f}")

## Phase 4 — Meta-Delta Generation

Needs **challenging held-out data** the model currently struggles with.
The meta-generator learns to produce corrective deltas from accumulated prediction errors.

In [ ]:
# Process Phase 4 data — challenging evaluation data
CACHE_PATH_P4 = f"{DATA_DIR}/phase4_challenge.pkl"

if os.path.exists(CACHE_PATH_P4):
    challenge_dataset = load_processed(CACHE_PATH_P4)
    challenge_dataset = CausalLMDataset(challenge_dataset)
else:
    print("Processing Phase 4 challenge data...")
    all_texts = []

    # MMLU — broad multi-domain knowledge
    print("  MMLU...")
    ds_mmlu = load_dataset("cais/mmlu", "all", split="test")
    mmlu_texts = [
        f"Question: {x['question']}\nChoices: {', '.join(x['choices'])}\nAnswer: {x['choices'][x['answer']]}"
        for x in ds_mmlu
    ]
    all_texts.extend(mmlu_texts)
    print(f"    → {len(mmlu_texts)} MMLU samples")

    # ARC Challenge — hard science reasoning
    print("  ARC Challenge...")
    ds_arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
    arc_texts = [
        f"Question: {x['question']}\nChoices: {', '.join(x['choices']['text'])}\nAnswer: {x['answerKey']}"
        for x in ds_arc
    ]
    all_texts.extend(arc_texts)
    print(f"    → {len(arc_texts)} ARC samples")

    # GSM8K test — held-out math reasoning
    print("  GSM8K test...")
    ds_gsm_test = load_dataset("openai/gsm8k", "main", split="test")
    gsm_test_texts = [f"Problem: {x['question']}\nSolution: {x['answer']}" for x in ds_gsm_test]
    all_texts.extend(gsm_test_texts)
    print(f"    → {len(gsm_test_texts)} GSM8K test samples")

    # MATH test — competition math
    print("  MATH test...")
    ds_math_test = load_dataset("lighteval/MATH", "all", split="test")
    math_test_texts = [f"Problem: {x['problem']}\nSolution: {x['solution']}" for x in ds_math_test]
    all_texts.extend(math_test_texts)
    print(f"    → {len(math_test_texts)} MATH test samples")

    import random
    random.shuffle(all_texts)
    print(f"Total: {len(all_texts)} challenge samples. Tokenizing...")

    challenge_dataset = CausalLMDataset.from_texts(all_texts, tokenizer, max_len=SEQ_LEN)
    save_processed(challenge_dataset.examples, CACHE_PATH_P4)

challenge_loader = DataLoader(challenge_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Phase 4 DataLoader ready: {len(challenge_dataset)} examples, {len(challenge_loader)} batches")

In [ ]:
# Phase 4 training — Meta-Delta Generation
from flx.training.phase4_meta import train_phase4

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase3.flx', device=DEVICE)
meta_gen = MetaDeltaGenerator(d_model=512, delta_rank=32, num_cortices=5)
model.attach_meta_generator(meta_gen)

history = train_phase4(
    model, meta_gen, challenge_loader,
    num_epochs=3, lr=1e-4,
    device=DEVICE, log_every=10,
)
save_flx(model, f'{FLX_HUB}/nano_phase4.flx')
accepted = sum(1 for h in history if h.get('accepted', 0) > 0)
print(f"Phase 4 done — {len(history)} steps, {accepted}/{len(history)} deltas accepted")

## Profiling (Experiment 2 — Thermal Efficiency)

In [ ]:
# Cell 10: FLOP profiling
from torch.profiler import profile, record_function, ProfilerActivity

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase2.flx', device=DEVICE)
# model.eval()
# input_ids = torch.randint(0, 32000, (1, 128), device=DEVICE)
#
# with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
#              record_shapes=True, with_flops=True) as prof:
#     with record_function('flx_forward'):
#         output = model(input_ids)
#
# print(prof.key_averages().table(sort_by='flops', row_limit=20))
print('Profiling: load a trained model and uncomment above')